# Image Motion Studio — GPU Worker

This notebook turns a free Google Colab session into the Image Motion Studio GPU worker. It:

1. Verifies the GPU
2. Sets your **CONVEX_HTTP_BASE**, **WORKER_TOKEN** and (optionally) **MODEL_ID**
3. Installs dependencies and loads **WAN 2.2 TI2V-5B** (diffusers)
4. Polls the studio for queued image-to-video jobs
5. Renders each job on the GPU and uploads the MP4 straight to Convex storage

Before running: issue a worker token in the studio (Settings → Worker token → Issue token) and paste it below together with your Convex HTTP base URL (Settings shows both values).

In [ ]:
#@title 1 · Configuration — paste your two studio values here

# From the studio Settings page → "Colab worker setup":
CONVEX_HTTP_BASE = "https://YOUR-DEPLOYMENT.convex.site"  #@param {type:"string"}
WORKER_TOKEN = "paste-your-worker-token-here"             #@param {type:"string"}

# Model to load. Leave as-is for the recommended free-Colab model.
# Any model id from the studio's MODELS registry works; ids marked as
# A14B/14B will NOT fit a free T4 — keep the 5B model unless you have an A100.
MODEL_ID = "wan2.2-ti2v-5b"

assert CONVEX_HTTP_BASE.startswith("https://"), "Set CONVEX_HTTP_BASE to your .convex.site URL"
assert len(WORKER_TOKEN) >= 16, "Paste the token issued in Settings (32+ chars)"

import requests, json, time

def api(path, payload):
    r = requests.post(f"{CONVEX_HTTP_BASE}/worker_api/{path}", json={"token": WORKER_TOKEN, **payload}, timeout=120)
    r.raise_for_status()
    return r.json()

# First contact — wrong token or wrong URL fails here, loudly.
health = api("health", {"status": "online", "message": "Colab worker starting"})
print("✓ authenticated —", CONVEX_HTTP_BASE, json.dumps(health))

In [ ]:
#@title 2 · GPU check
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime → Change runtime type → select a GPU (free T4 is fine).")

GPU_NAME = torch.cuda.get_device_name(0)
FREE, TOTAL = torch.cuda.mem_get_info()
VRAM_GB = round(TOTAL / 1024**3, 1)
print(f"GPU: {GPU_NAME} · {VRAM_GB} GB VRAM · {round(FREE/1024**3,1)} GB free")

api("health", {"status": "online", "gpuName": GPU_NAME, "vramGb": VRAM_GB, "message": "GPU ready"})
print("✓ GPU reported to the studio")

In [ ]:
#@title 3 · Install dependencies (first run only, ~3–5 minutes)
%pip -q install diffusers transformers accelerate imageio[ffmpeg] safetensors
import diffusers, transformers
print("diffusers", diffusers.__version__, "· transformers", transformers.__version__)

In [ ]:
#@title 4 · Load the video model (first run only, ~2–4 minutes)
import torch
from diffusers import WanPipeline  # TI2V-5B ships as a standard Wan pipeline

MODEL_REPOS = {
    "wan2.2-ti2v-5b": "Wan-AI/Wan2.2-TI2V-5B-Diffusers",
    "wan2.1-t2v-1.3b": "Wan-AI/Wan2.1-T2V-1.3B-Diffusers",
}

if MODEL_ID not in MODEL_REPOS:
    raise ValueError(
        f"{MODEL_ID} is not loadable on free Colab. "
        f"Pick one of {list(MODEL_REPOS)} — the A14B models need A100-class VRAM."
    )

MODEL_REPO = MODEL_REPOS[MODEL_ID]
print("loading", MODEL_REPO, "…")

pipe = WanPipeline.from_pretrained(
    MODEL_REPO,
    torch_dtype=torch.bfloat16,
)
pipe.to("cuda")
pipe.enable_vae_tiling()          # big VRAM savings, negligible quality cost
try:
    pipe.enable_vae_slicing()
except Exception:
    pass

api("health", {"status": "online", "gpuName": GPU_NAME, "vramGb": VRAM_GB,
               "loadedModel": MODEL_ID, "message": "Model loaded — polling for jobs"})
print("✓ model loaded and reported to the studio")

In [ ]:
#@title 5 · Worker loop — claims jobs, renders, uploads MP4s. Keep this cell running.
import io, time, torch, imageio
import PIL.Image
from PIL import Image

HEARTBEAT_SECONDS = 60
CLAIM_INTERVAL_SECONDS = 15

def report_progress(key, status, progress, message):
    try:
        api("progress", {"workerJobKey": key, "status": status,
                         "progress": progress, "message": message})
    except Exception as e:
        print("  (progress update failed, continuing):", e)

def check_cancelled(key):
    try:
        out = api("job-status", {"workerJobKey": key})
        return bool(out.get("cancelled"))
    except Exception:
        return False

def upload_bytes(upload_url, data: bytes, content_type: str) -> str:
    r = requests.post(upload_url, data=data,
                      headers={"Content-Type": content_type}, timeout=600)
    r.raise_for_status()
    return r.json()["storageId"]

# Aspect-ratio aware frame sizing (WAN TI2V-5B works at multiples of 16).
def frame_size(aspect: str, resolution: str):
    short = 480 if resolution == "480p" else 720
    def snap(n): return int(round(n / 16)) * 16
    if aspect == "9:16":  return snap(short), snap(short * 16 / 9)
    if aspect == "1:1":   return snap(short), snap(short)
    return snap(short * 16 / 9), snap(short)  # 16:9 default

def render_job(job) -> bytes:
    """Runs REAL image-to-video inference. No placeholders, no sample output."""
    s = job["settings"]
    width, height = frame_size(s.get("aspectRatio", "16:9"), s.get("resolution", "480p"))

    # 1 · download the source image and make it the first frame
    img = Image.open(io.BytesIO(requests.get(job["imageUrl"], timeout=120).content)).convert("RGB")
    img = img.resize((width, height), Image.LANCZOS)

    # 2 · video generation (this is the actual GPU work)
    generator = torch.Generator("cuda")
    if s.get("seed") is not None:
        generator = generator.manual_seed(int(s["seed"]))

    steps = 20                      # TI2V-5B quality/runtime trade-off for Colab
    total = steps + 12              # + encode/upload bookkeeping for progress math
    done = 0
    def bump(p, msg):
        nonlocal done
        done = p
        report_progress(job["workerJobKey"], "generating",
                        min(95, round(p / total * 100)), msg)

    report_progress(job["workerJobKey"], "loading_model", 2, "preparing tensors")

    result = pipe(
        image=img,
        prompt=job["prompt"],
        negative_prompt=job.get("negativePrompt") or None,
        width=width,
        height=height,
        num_frames=81,              # TI2V-5B native clip length (~5 s at 16 fps)
        num_inference_steps=steps,
        guidance_scale=5.0,
        generator=generator,
        callback_on_step_end=lambda p, i, t, kw: (bump(i + 1, f"diffusion step {i+1}/{steps}"), kw)[1],
    )
    frames = result.frames[0]       # list[PIL.Image]

    # 3 · encode a real MP4 with ffmpeg via imageio
    report_progress(job["workerJobKey"], "processing", 90, "encoding MP4")
    buf = io.BytesIO()
    imageio.mimwrite(buf, frames, format="FFMPEG", codec="libx264",
                     fps=16, quality=8, pixelformat="yuv420p",
                     macro_block_size=16, output_params=["-movflags", "+faststart"])
    return buf.getvalue()

print("worker running — claim/heartbeat loop started")
print("queue a job in the studio; this cell picks it up automatically")
print("stop the cell (⏹) or Runtime → Interrupt to take the worker offline")

last_heartbeat = 0.0
while True:
    now = time.time()
    try:
        if now - last_heartbeat > HEARTBEAT_SECONDS:
            api("health", {"status": "online", "gpuName": GPU_NAME, "vramGb": VRAM_GB,
                           "loadedModel": MODEL_ID, "message": "polling for jobs"})
            last_heartbeat = now

        claim = api("claim", {})
        job = claim.get("job")
        if not job:
            time.sleep(CLAIM_INTERVAL_SECONDS)
            continue

        print("▸ job", job["workerJobKey"][:8], "·", job["model"])
        report_progress(job["workerJobKey"], "connecting", 1, "claimed — downloading source image")

        if check_cancelled(job["workerJobKey"]):
            print("  job was cancelled before render — skipping")
            continue

        try:
            mp4 = render_job(job)

            if check_cancelled(job["workerJobKey"]):
                print("  job was cancelled during render — discarding")
                continue

            video_storage_id = upload_bytes(job["videoUploadUrl"], mp4, "video/mp4")
            out = api("complete", {"workerJobKey": job["workerJobKey"],
                                   "videoStorageId": video_storage_id})
            print("  ✓ completed — videoId", out.get("videoId", "?"))

        except KeyboardInterrupt:
            raise
        except Exception as e:
            msg = str(e)[:500]
            print("  ✗ job failed:", msg)
            try:
                api("fail", {"workerJobKey": job["workerJobKey"], "error": msg})
            except Exception as e2:
                print("  (fail report also failed):", e2)

    except KeyboardInterrupt:
        print("worker stopped by user")
        try:
            api("health", {"status": "error", "message": "worker stopped"})
        except Exception:
            pass
        break
    except Exception as e:
        # transient network errors: wait and retry, the studio shows truth
        print("loop error (retrying):", str(e)[:200])
        time.sleep(10)